# 01 — Data sanitisation

**Input:** `raw-data/video_actions.csv` (raw event log, ~363 MB, 1,040,534 rows).

**Outputs (in `artefacts/`):**
- `clean_events.parquet` — engaged-cohort events for the five target courses (ping-free, spike-free).
- `cohort_engaged.parquet` — registrations passing the 20% lecture-coverage filter, with their coverage value.
- numbers backing Table 5.1 (per-course breakdown) and Table 5.2 (action-type distribution).

Methodology reference: `thesis-latex/content/05_methodology.tex` §5.1.

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = Path('raw-data')
OUT = Path('artefacts')
OUT.mkdir(exist_ok=True)

EVENTS_CSV          = RAW / 'video_actions.csv'
TARGET_COURSE_IDS   = [1, 5, 30, 32, 42]
COVERAGE_THRESHOLD  = 0.20
SPIKE_SIGMA         = 5  # drop registrations with event count > mean + 5*std

## Load raw events

In [2]:
raw = pd.read_csv(EVENTS_CSV, low_memory=False)
print(f'raw rows: {len(raw):,}')
raw.action_type.value_counts()

raw rows: 1,040,534


action_type
ping          811093
play           72416
pause          65646
advance        47196
rewind         32172
end             8849
playback        3092
text_track        70
Name: count, dtype: int64

## Sanitisation

1. Restrict to the five target courses.
2. Drop ping events (~76% of the raw feed in the target-5 subset; do not carry interaction signal).
3. Drop spike registrations (event count above mean + 5σ — typically automated/stuck-client sessions).
4. Keep only registrations whose lecture coverage strictly exceeds 20% (engaged-learner cohort).

In [3]:
df = raw[raw.course_id.isin(TARGET_COURSE_IDS) & (raw.action_type != 'ping')].copy()
ping_share_target = (raw[raw.course_id.isin(TARGET_COURSE_IDS)].action_type == 'ping').mean() * 100
print(f'after target-{len(TARGET_COURSE_IDS)} + ping drop: {len(df):,} ({ping_share_target:.1f}% of target-5 raw was ping)')

after target-5 + ping drop: 172,740 (75.7% of target-5 raw was ping)


In [4]:
ec = df.registration_id.value_counts()
spike_threshold = ec.mean() + SPIKE_SIGMA * ec.std()
spike_regs = ec[ec > spike_threshold].index.tolist()
print(f'spike threshold (mean + {SPIKE_SIGMA}σ): {spike_threshold:.0f}')
print(f'spike registrations dropped: {spike_regs}')
df = df[~df.registration_id.isin(spike_regs)].copy()
print(f'after spike drop: {len(df):,}')

spike threshold (mean + 5σ): 2118
spike registrations dropped: [5138, 6838, 6454, 6500]
after spike drop: 160,902


In [5]:
lectures_per_course = df.groupby('course_id').lecture_id.nunique().to_dict()

reg_course = (
    df.groupby(['registration_id', 'course_id'])
      .lecture_id.nunique()
      .rename('reached')
      .reset_index()
)
reg_course['total']    = reg_course.course_id.map(lectures_per_course)
reg_course['coverage'] = reg_course.reached / reg_course.total

engaged = reg_course[reg_course.coverage > COVERAGE_THRESHOLD]
print(f'engaged registrations (coverage > {COVERAGE_THRESHOLD}): {len(engaged)}')

engaged_set = set(engaged.registration_id)
clean = df[df.registration_id.isin(engaged_set)].copy()
print(f'events in engaged cohort: {len(clean):,}')

engaged registrations (coverage > 0.2): 405
events in engaged cohort: 146,118


## Persist

In [6]:
clean.to_parquet(OUT / 'clean_events.parquet', index=False)
engaged.to_parquet(OUT / 'cohort_engaged.parquet', index=False)
print(f'wrote {OUT / "clean_events.parquet"} and {OUT / "cohort_engaged.parquet"}')

wrote artefacts/clean_events.parquet and artefacts/cohort_engaged.parquet


## Table 5.1 — engaged-cohort breakdown

In [7]:
table51 = clean.groupby('course_id').agg(
    events=('id', 'count'),
    users=('user_id', 'nunique'),
    regs=('registration_id', 'nunique'),
    lectures=('lecture_id', 'nunique'),
).reindex(TARGET_COURSE_IDS)
totals = pd.DataFrame({
    'events':    [len(clean)],
    'users':     [clean.user_id.nunique()],
    'regs':      [clean.registration_id.nunique()],
    'lectures':  [clean.lecture_id.nunique()],
}, index=['Total'])
print(pd.concat([table51, totals]))

       events  users  regs  lectures
1       48256    115   116        31
5       20469     59    60        37
30      34992     88    88        29
32      27445     47    47        36
42      14956     94    94        29
Total  146118    339   405       162


## Table 5.2 — action-type distribution

In [8]:
ac = clean.action_type.value_counts()
table52 = pd.DataFrame({'count': ac, 'pct': (ac / len(clean) * 100).round(1)})
print(table52)

             count   pct
action_type             
play         46793  32.0
pause        41979  28.7
advance      27354  18.7
rewind       21102  14.4
end           6836   4.7
playback      2027   1.4
text_track      27   0.0
